# 🏠 House Price Prediction with Linear Regression (from scratch)

A hands-on implementation of the **Linear Regression with One Variable** lecture from Andrew Ng's Machine Learning Specialization:

$$f_{w,b}(x) = wx + b$$

**What we'll do**
1. Load a real house-price dataset.
2. Explore it and pick **one feature** ($x$).
3. Write the model $f(x) = wx + b$ ourselves.
4. Write a **cost function** that measures how wrong the model is.
5. Write **gradient descent** to find the best $w$ and $b$.
6. Evaluate the model on data it has never seen.
7. Compare with scikit-learn.
8. (Bonus) Use all features: **multiple linear regression**.

> Run the cells in order with **Shift + Enter**, or use **Runtime → Run all**.

## Step 1: Import the libraries

- **numpy**: fast math on arrays.
- **pandas**: tables (DataFrames).
- **matplotlib**: plotting.
- **scikit-learn**: the dataset loader, train/test splitting and reference models.

In [ ]:
import numpy as np                 # array math ("np" is the standard alias)
import pandas as pd                # tables (DataFrame)
import matplotlib.pyplot as plt    # plotting

# "from ... import ..." imports a single function from a library
from sklearn.datasets import fetch_california_housing   # dataset loader
from sklearn.model_selection import train_test_split    # train/test split

plt.rcParams["figure.figsize"] = (8, 5)   # default size for every plot
np.random.seed(42)                        # fix randomness so results are reproducible

## Step 2: Load the data

We use the **California Housing dataset**: real data from the 1990 US Census, with about **20,640 block groups** (small neighborhoods, not single houses).
scikit-learn ships the *loader function*; the first time you call it, it downloads the data file (a few seconds) and caches it.

**Features**

| Column | Meaning |
|---|---|
| `MedInc` | Median income in the block (in tens of thousands of USD) |
| `HouseAge` | Median house age |
| `AveRooms` | Average number of rooms per house |
| `AveBedrms` | Average number of bedrooms per house |
| `Population` | Block population |
| `AveOccup` | Average number of people per house |
| `Latitude` / `Longitude` | Location |

**Target ($y$):** `MedHouseVal`, the median house value in **hundreds of thousands of USD** (2.5 means 250,000 USD).

In [ ]:
# Downloads the file the first time, then reads it from the local cache.
# as_frame=True -> return a pandas DataFrame with column names
data = fetch_california_housing(as_frame=True)

df = data.frame   # the full table: 8 features + the price column

# First 5 rows. The last expression in a cell is displayed without print()
df.head()

## Step 3: Explore the data (EDA)

Before building any model, check the size of the data, whether anything is missing, and what the numbers look like.

In [ ]:
# shape is an attribute (not a function, so no parentheses) -> (rows, columns)
print("Rows, columns:", df.shape)

# isnull() -> True for every missing cell; sum() counts them (True = 1)
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Summary statistics: count, mean, std, min, quartiles, max
df.describe().round(2)

### Price distribution

Notice the tall bar at **5.0**. Every house worth more than 500,000 USD was recorded as exactly 500,000 USD (this is called **capping**).
We don't know the real price of those houses, so they would mislead the model. We remove them in the next cell.

In [ ]:
# Histogram: split prices into 50 bins and count the houses in each bin
plt.hist(df["MedHouseVal"], bins=50, color="steelblue", edgecolor="white")
plt.xlabel("Price ($100k)")
plt.ylabel("Count")
plt.title("Distribution of house prices")
plt.show()

In [ ]:
before = len(df)

# Boolean indexing:
#   df["MedHouseVal"] < 5.0  -> a column of True/False
#   df[...]                  -> keeps only the rows marked True
#   .copy()                  -> an independent copy (avoids SettingWithCopyWarning)
df = df[df["MedHouseVal"] < 5.0].copy()

# f-string: any expression inside {} is evaluated and inserted into the text
print(f"Removed {before - len(df)} rows, {len(df)} rows left")

## Step 4: Choose the feature ($x$)

This is **univariate** regression, so we use **one feature only**.
We pick the feature with the strongest **correlation** with price.

- Correlation is a number between -1 and 1.
- Close to **1**: when $x$ goes up, $y$ goes up (strong positive relationship).
- Close to **0**: no linear relationship.

In [ ]:
# df.corr()            -> correlation between every pair of columns (9x9 table)
# ["MedHouseVal"]      -> how each feature relates to price
# .drop("MedHouseVal") -> remove price vs. itself (always 1)
# .sort_values(...)    -> sort from largest to smallest
corr = df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)
print(corr.round(3))

corr.plot(kind="barh", color="teal")           # barh = horizontal bars
plt.title("Correlation with house price")
plt.axvline(0, color="black", linewidth=0.8)   # vertical line at zero
plt.show()

**`MedInc` (income) is by far the strongest.** That makes sense: neighborhoods with higher incomes have more expensive houses.

So:
- $x$ = `MedInc`
- $y$ = `MedHouseVal`

Let's plot them, just like the red crosses in the lecture:

In [ ]:
# .values converts a pandas column into a plain numpy array
x = df["MedInc"].values        # input (feature)
y = df["MedHouseVal"].values   # correct answer (target)

# scatter: one dot per block. s = dot size, alpha = transparency (0 invisible, 1 solid)
plt.scatter(x, y, s=3, alpha=0.3, color="red")
plt.xlabel("Median income ($10k)  ->  x")
plt.ylabel("House price ($100k)  ->  y")
plt.title("Training data")
plt.show()

## Step 5: Split the data (train / test)

- **Training set (80%)**: the model learns from it.
- **Test set (20%)**: kept hidden from the model and used only at the end.

Why? To check whether the model actually **learned** the relationship or just **memorized** the data, like testing a student on new questions instead of the ones in the textbook.

In [ ]:
# Shuffles the data, splits it, and returns 4 arrays in this order (tuple unpacking)
# test_size=0.2   -> 20% goes to the test set
# random_state=42 -> the same split every time you run it
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

m = len(x_train)   # number of training examples (same "m" as in the lecture)
print("Training examples (m):", m)
print("Test examples:", len(x_test))

## Step 6: The model $f_{w,b}(x) = wx + b$

Exactly the diagram from the lecture:

$x \rightarrow \boxed{f} \rightarrow \hat{y}$

- $w$: the slope (how much the price changes when income goes up by 1).
- $b$: where the line crosses the $y$ axis.
- $\hat{y}$ ("y-hat"): the **predicted** price; $y$ is the **actual** price.

In [ ]:
def predict(x, w, b):
    # f_wb(x) = w * x + b
    # Vectorization: if x is an array of thousands of numbers, numpy computes
    # a prediction for every one at once, with no loop.
    # If x is a single number, it returns a single number.
    return w * x + b

### Try your own $w$ and $b$

Before letting the computer learn, try it yourself: change `w` and `b` in this cell, run it a few times, and try to make the blue line pass through the middle of the red dots.

You are doing **by hand** what the learning algorithm will do **automatically**.

In [ ]:
w, b = 0.3, 0.5   # <- change these and re-run

plt.scatter(x_train, y_train, s=3, alpha=0.3, color="red", label="Actual (y)")

# linspace: 100 evenly spaced values from the smallest to the largest income
# (plotting x_train directly would zig-zag because it isn't sorted)
x_line = np.linspace(x_train.min(), x_train.max(), 100)
plt.plot(x_line, predict(x_line, w, b), color="blue", linewidth=3, label=f"f(x) = {w}x + {b}")
plt.xlabel("Median income ($10k)")
plt.ylabel("House price ($100k)")
plt.legend()
plt.show()

## Step 7: The cost function

We need **one number** that tells us how good or bad a line is. That's the **cost function** (covered in the next lecture):

$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} \left( f_{w,b}(x^{(i)}) - y^{(i)} \right)^2$$

**In plain words**
1. For each house: error = predicted price − actual price.
2. Square the error (so positive and negative errors don't cancel, and big errors are punished more).
3. Average over all houses and divide by 2 (the 2 only makes the derivative cleaner).

**The lower $J$, the better the model.**

In [ ]:
def compute_cost(x, y, w, b):
    m = len(x)                            # number of examples
    y_hat = predict(x, w, b)              # y-hat: a prediction for every house
    errors = y_hat - y                    # error for every house (element-wise)
    cost = np.sum(errors ** 2) / (2 * m)  # square -> sum -> divide by 2m
    return cost

#   Formula             Code
#   f(x_i) - y_i        errors
#   (...)^2             ** 2
#   sum                 np.sum
#   1 / 2m              / (2 * m)

print("Cost with w=0.3, b=0.5:", round(compute_cost(x_train, y_train, 0.3, 0.5), 4))
print("Cost with w=0.4, b=0.5:", round(compute_cost(x_train, y_train, 0.4, 0.5), 4))
print("Cost with w=0.0, b=0.0:", round(compute_cost(x_train, y_train, 0.0, 0.0), 4))

## Step 8: Gradient descent (how the model learns)

Imagine standing on a hill in the dark, trying to reach the lowest point (lowest cost). You feel the ground, find which direction goes downhill, take a step, and repeat.

**Update rules**

$$w = w - \alpha \frac{\partial J}{\partial w} \qquad b = b - \alpha \frac{\partial J}{\partial b}$$

where

$$\frac{\partial J}{\partial w} = \frac{1}{m} \sum_{i=1}^{m} \left(f_{w,b}(x^{(i)}) - y^{(i)}\right) x^{(i)} \qquad \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \left(f_{w,b}(x^{(i)}) - y^{(i)}\right)$$

- **The gradient** points in the direction where the cost goes **up**, so we step the **opposite** way (hence the minus sign).
- **$\alpha$ (learning rate)** is the step size:
  - too small: it gets there, but very slowly;
  - too large: it overshoots and may never converge (the cost can even increase).

In [ ]:
def compute_gradient(x, y, w, b):
    m = len(x)
    errors = predict(x, w, b) - y
    dj_dw = np.sum(errors * x) / m   # dJ/dw: has "* x" because w is multiplied by x
    dj_db = np.sum(errors) / m       # dJ/db: the derivative of b alone is 1
    # Divided by m (not 2m): the 2 from the square cancels the 2 in 1/(2m)
    return dj_dw, dj_db              # returns two values


def gradient_descent(x, y, w_init, b_init, alpha, num_iters):
    w, b = w_init, b_init
    cost_history = []            # store the cost after every step so we can plot it

    for i in range(num_iters):   # range(10000) -> 0 .. 9999
        # 1) compute BOTH derivatives using the current values
        dj_dw, dj_db = compute_gradient(x, y, w, b)

        # 2) update both together (simultaneous update): a step of size alpha downhill
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        cost_history.append(compute_cost(x, y, w, b))   # append = add to end of list

        # Print only every 1/10 of the run:  // integer division,  % remainder
        # {i:5d} integer in 5 spaces,  {w:.4f} 4 decimals,  [-1] last element
        if i % (num_iters // 10) == 0:
            print(f"Iteration {i:5d} | Cost: {cost_history[-1]:.4f} | w: {w:.4f} | b: {b:.4f}")

    return w, b, cost_history

### Train it 🚀

Start from $w = 0, b = 0$ (a flat line on the floor) and let the algorithm improve it.
Watch the **cost** go down.

In [ ]:
alpha = 0.01
iterations = 10000

w_final, b_final, cost_history = gradient_descent(x_train, y_train, 0, 0, alpha, iterations)

print(f"\n✅ Result: w = {w_final:.4f}, b = {b_final:.4f}")
print(f"   Model: f(x) = {w_final:.4f} * x + {b_final:.4f}")

### Learning curve

If the cost **drops and then flattens**, gradient descent worked and **converged**.
If it rises or oscillates, the learning rate is too large: make it smaller.

In [ ]:
plt.plot(cost_history, color="purple")   # one list -> x axis is the iteration number
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.title("Cost vs. iterations")
plt.show()

### The final line

This is the blue line from the lecture, except this time the computer found it.

In [ ]:
plt.scatter(x_train, y_train, s=3, alpha=0.3, color="red", label="Actual price (y)")
plt.plot(x_line, predict(x_line, w_final, b_final), color="blue", linewidth=3,
         label=f"f(x) = {w_final:.3f}x + {b_final:.3f}")
plt.xlabel("Median income ($10k)")
plt.ylabel("House price ($100k)")
plt.title("Linear regression fit")
plt.legend()
plt.show()

# Interpreting w: income is in units of 10k USD and price in units of 100k USD
print(f"Every extra 10,000 USD of median income adds about "
      f"{w_final * 100_000:,.0f} USD to the predicted house price.")

## Step 9: Evaluate on the test set

Now test the model on houses it has **never seen**:

- **MAE (mean absolute error)**: the average error, easy to read ("the model is off by about X dollars").
- **RMSE (root mean squared error)**: like MAE, but punishes big errors more.
- **R² (R-squared)**: from 0 to 1, the **share of the variation in prices** the model explains. 1 = perfect, 0 = no better than always predicting the average.

In [ ]:
# Use the learned w and b on the test data the model never saw
y_pred = predict(x_test, w_final, b_final)

mae  = np.mean(np.abs(y_pred - y_test))            # mean absolute error
rmse = np.sqrt(np.mean((y_pred - y_test) ** 2))    # square root brings it back to price units
# R^2 = 1 - (our model's errors) / (errors of a model that always predicts the mean)
r2   = 1 - np.sum((y_test - y_pred) ** 2) / np.sum((y_test - y_test.mean()) ** 2)

# 100_000 == 100000 (the _ is only for readability);  :,.0f -> thousands separators, no decimals
print(f"MAE : {mae:.3f}  ->  the model is off by about ${mae * 100_000:,.0f} on average")
print(f"RMSE: {rmse:.3f}  ->  ${rmse * 100_000:,.0f}")
print(f"R²  : {r2:.3f}  ->  income alone explains about {r2 * 100:.0f}% of the variation in prices")

### Predicted vs. actual

If the model were perfect, every dot would sit on the dashed line.

In [ ]:
plt.scatter(y_test, y_pred, s=4, alpha=0.3)
lims = [0, 5]
plt.plot(lims, lims, "k--", label="Perfect prediction")
plt.xlabel("Actual price (y)")
plt.ylabel("Predicted price (ŷ)")
plt.title("Predicted vs. actual (test set)")
plt.legend()
plt.show()

## Step 10: Compare with scikit-learn

In practice nobody writes gradient descent by hand; a library does it in two lines.
If our code is correct, we should get **almost the same $w$ and $b$**.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()   # a new, untrained model object

# fit = train. scikit-learn expects features as a 2D table (rows x columns):
# reshape(-1, 1) turns shape (m,) into (m, 1); -1 means "work this dimension out yourself"
model.fit(x_train.reshape(-1, 1), y_train)

# coef_ = array of weights (w), intercept_ = b
# A trailing "_" is the scikit-learn convention for values learned during training
print(f"scikit-learn: w = {model.coef_[0]:.4f}, b = {model.intercept_:.4f}")
print(f"from scratch: w = {w_final:.4f}, b = {b_final:.4f}")
print(f"\nR² (scikit-learn) on the test set: {model.score(x_test.reshape(-1, 1), y_test):.3f}")

## Step 11: Predict the price of a new house 🏡

Change the income value and re-run:

In [ ]:
new_income = 6.0   # i.e. a median income of 60,000 USD  <- change it

price = predict(new_income, w_final, b_final)   # one number in -> one prediction out
print(f"Neighborhood median income: ${new_income * 10_000:,.0f}")
print(f"-> Predicted house price:    ${price * 100_000:,.0f}")

## 🎁 Bonus: multiple linear regression (Week 2)

Income alone explains less than half of the story. What if we use **all the features**?

$$f_{\vec{w},b}(\vec{x}) = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$$

**Feature scaling.** The features have very different ranges (`Population` is in the thousands, `AveBedrms` is around 1), which makes gradient descent slow.
`StandardScaler` gives every feature a mean of 0 and a standard deviation of 1. (You'll cover this in Week 2.)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Capital X = a 2D matrix with all 8 features (same convention as the course)
feature_names = df.drop(columns="MedHouseVal").columns
X = df[feature_names].values
y_all = df["MedHouseVal"].values

# Same random_state and same rows -> the same split, so the comparison is fair
X_train, X_test, Y_train, Y_test = train_test_split(X, y_all, test_size=0.2, random_state=42)

# StandardScaler: (value - column mean) / column standard deviation
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit: learn mean and std from the TRAIN set only
X_test_s  = scaler.transform(X_test)        # apply the same numbers to the test set (no data leakage)

multi = LinearRegression().fit(X_train_s, Y_train)   # fit() returns the trained model
y_pred_multi = multi.predict(X_test_s)

r2_multi  = multi.score(X_test_s, Y_test)            # score() returns R² directly
mae_multi = np.mean(np.abs(y_pred_multi - Y_test))

print(f"One feature (MedInc): R² = {r2:.3f} | MAE = ${mae * 100_000:,.0f}")
print(f"All 8 features      : R² = {r2_multi:.3f} | MAE = ${mae_multi * 100_000:,.0f}")

In [ ]:
# coef_ holds 8 weights; put them in a Series labeled with the feature names
weights = pd.Series(multi.coef_, index=feature_names).sort_values()

# List comprehension: red for negative weights, green for positive ones
weights.plot(kind="barh", color=["tomato" if v < 0 else "seagreen" for v in weights])
plt.title("Learned weights (scaled features)")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()

**Reading the weights:** green pushes the price up, red pushes it down. `Latitude` and `Longitude` are strongly negative because the most expensive houses in California are near the coast (San Francisco and Los Angeles).

> Because the features are correlated with each other, individual weights can be misleading on their own (for example, `AveRooms` and `AveBedrms` often get opposite signs). Look at them as a rough guide, not exact effects.

---

## 📝 Summary

| Concept | Where it appears |
|---|---|
| Training set (features + targets) | Steps 2–5 |
| Model $f_{w,b}(x) = wx + b$ | Step 6 |
| $y$ vs. $\hat{y}$ | Steps 6 and 9 |
| Cost function $J(w,b)$ | Step 7 |
| Gradient descent and learning rate | Step 8 |
| Univariate vs. multiple regression | Step 8 vs. the bonus |

## 🧪 Exercises

1. Set `alpha` to **0.1**, then **0.0001**, and see what happens to the cost curve.
2. Swap `MedInc` for `AveRooms` and compare R².
3. Keep the capped houses (price 5.0) and retrain. Does the result get better or worse?
4. Set `iterations` to just 100. Does the model converge?

## 🖥️ Step 12: A simple front end (Gradio)

Now let's wrap both models in a small web app: you type the inputs, press **Predict**, and get a price.

**Gradio** builds the interface from a Python function: inputs go in, the return value is shown as output.
When it launches in Colab, the app appears right below the cell, plus a public link (`*.gradio.live`) you can open in a new tab or send to a friend. The link works for about a week while this notebook's runtime is running.

> Run all the cells above first: the app uses `w_final`, `b_final`, `multi` and `scaler` that we trained.

In [ ]:
# Gradio isn't always preinstalled on Colab; -q = quiet install
!pip install -q gradio

In [ ]:
import gradio as gr

# Slider ranges come from the data itself: 1st to 99th percentile, default = median.
# This keeps users inside the range the model was trained on.
def slider(col, label, step):
    lo, hi = df[col].quantile([0.01, 0.99])
    return gr.Slider(float(lo), float(hi), value=float(df[col].median()), step=step, label=label)


def predict_price(med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, lat, lon):
    # 1) Our from-scratch model: only uses income, f(x) = w*x + b
    simple = predict(med_inc, w_final, b_final)

    # 2) The multiple regression model: needs all 8 features,
    #    in the SAME order as training, scaled with the SAME scaler
    row = np.array([[med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, lat, lon]])
    full = multi.predict(scaler.transform(row))[0]

    # Prices are in units of 100k USD; clip to the training range (we removed 5.0+)
    simple = float(np.clip(simple, 0.15, 5.0))
    full = float(np.clip(full, 0.15, 5.0))

    return (f"${simple * 100_000:,.0f}",
            f"${full * 100_000:,.0f}")


# gr.Blocks lets us lay out the page ourselves (rows, columns, buttons)
with gr.Blocks(title="California House Price Predictor") as demo:
    gr.Markdown("# 🏠 California House Price Predictor\n"
                "Describe a neighborhood and press **Predict**. "
                "Income is in tens of thousands of USD (6 = 60,000 USD).")

    with gr.Row():
        with gr.Column():
            inc   = slider("MedInc", "Median income (x 10k USD)", 0.1)
            age   = slider("HouseAge", "Median house age (years)", 1)
            rooms = slider("AveRooms", "Average rooms per house", 0.1)
            beds  = slider("AveBedrms", "Average bedrooms per house", 0.05)
        with gr.Column():
            pop   = slider("Population", "Block population", 10)
            occ   = slider("AveOccup", "Average people per house", 0.1)
            lat   = slider("Latitude", "Latitude", 0.01)
            lon   = slider("Longitude", "Longitude", 0.01)

    btn = gr.Button("Predict", variant="primary")

    with gr.Row():
        out_simple = gr.Textbox(label="Model 1: income only (our gradient descent)")
        out_full   = gr.Textbox(label="Model 2: all 8 features (scikit-learn)")

    inputs = [inc, age, rooms, beds, pop, occ, lat, lon]
    # When the button is clicked: call predict_price(inputs) and show the results in the outputs
    btn.click(predict_price, inputs=inputs, outputs=[out_simple, out_full])

    # Ready-made examples: click one to fill all the sliders
    gr.Examples(
        examples=[
            [8.0, 30, 6.0, 1.0, 1000, 2.5, 37.77, -122.42],   # wealthy, San Francisco
            [3.5, 25, 5.0, 1.1, 1500, 3.2, 34.05, -118.25],   # average, Los Angeles
            [2.5, 20, 5.5, 1.1, 1200, 3.0, 36.74, -119.79],   # lower income, inland (Fresno)
        ],
        inputs=inputs,
        label="Try an example",
    )

# share=True creates the public *.gradio.live link
demo.launch(share=True, debug=False)

### How it works

| Part | What it does |
|---|---|
| `predict_price(...)` | The "back end": takes the 8 inputs, returns two formatted prices |
| `gr.Slider` | An input; its range comes from the training data |
| `gr.Button` + `.click(...)` | Calls the function when pressed |
| `gr.Textbox` | Shows the result |
| `scaler.transform(row)` | New inputs must be scaled exactly like the training data, or the predictions are wrong |
| `demo.launch(share=True)` | Starts the web server and prints the public link |

**Things to try:** keep income the same and move only `Latitude`/`Longitude` between San Francisco and an inland city. Model 1 won't change at all (it only knows income), while Model 2 will.

To stop the app, press the **stop** button on the cell (or **Runtime → Interrupt execution**).